# Project 8 -- Vinay Gannamaneni

**TA Help:** (for instance) John Smith, Alice Jones, etc., list names of any TAs who helped you

- For example: Help with figuring out how to write a function (describe the tasks that they helped you with)

**Collaboration:** My Friend in CS, My Uncle, Another Student, etc., list names of any other people who helped you

(describe the tasks that they helped you with)
- For example: helped figuring out how to load the dataset.
- Another example: helped debug error with my plot.

**Internet Resources:** Stack Exchange, Stack Overflow, etc.

(describe any information that you learned from internet resources, including the URLs)
- data frames in Pandas versus R from StackOverflow  https://stackoverflow.com/questions/8991709/why-were-pandas-merges-in-python-faster-than-data-table-merges-in-r-in-2012

**ChatGPT, Gemini, Claude, etc:** Any language models or generative AI chatbots that helped you.

(if you used any such tools, please tell us here)
- For example:  I asked ChatGPT how to define a new data frames
- Another example:  Gemini told me how to make a function for sorting my data

- ***Link to AI Chat History***: Please share a link to your chat if you used AI (ex. ChatGPT Shared Links)
**OVERALL MESSAGE:** Any time that you used anything except your brain to solve the questions in these projects, you need to disclose such resources at the start of the project, with details about your usage of the tools.

**YOUR OWN WORK:** Even when you utilize other resources, do NOT just copy and paste.  Write all explanations in your own words, using several sentences in English, which are understandable and which you wrote (and did not just copy and paste).

## Custom functions to filter songs by energy level and by album

In [3]:
import pandas as pd

pd.set_option('display.max_columns', None)
ts_songs = pd.read_csv('/anvil/projects/tdm/data/spotify/taylor_swift_discography_updated.csv', sep=";")

Setting display options and loading in data set

To find min and max levels

In [5]:
max_energy = ts_songs['energy'].max()
min_energy = ts_songs['energy'].min()
print(f"Maximum energy level: {max_energy}")
print(f"Minimum energy level: {min_energy}")

Maximum energy level: 0.95
Minimum energy level: 0.118


In [6]:
def find_songs_with_energy(input_df, threshold):
    my_output = input_df[input_df["energy"] >= threshold]
    return my_output

my_median = ts_songs['energy'].median()
high_energy_df = find_songs_with_energy(ts_songs, my_median)

print(f"Number of high energy songs (>= {my_median}): {len(high_energy_df)}")

Number of high energy songs (>= 0.57): 289


Define energy filter function and calculte median to test function

In [7]:
def find_songs_by_album(input_df, album_name):
    my_output = input_df[input_df["album"] == album_name]
    return my_output

ttpd_songs = find_songs_by_album(ts_songs, "The Tortured Poets Department: The Anthology")
print(f"Songs in TTPD Anthology: {len(ttpd_songs)}")
print(ttpd_songs[['track_name', 'album']].head())

Songs in TTPD Anthology: 31
                             track_name  \
0         Fortnight (feat. Post Malone)   
1         The Tortured Poets Department   
2  My Boy Only Breaks His Favorite Toys   
3                              Down Bad   
4                       So Long, London   

                                          album  
0  The Tortured Poets Department: The Anthology  
1  The Tortured Poets Department: The Anthology  
2  The Tortured Poets Department: The Anthology  
3  The Tortured Poets Department: The Anthology  
4  The Tortured Poets Department: The Anthology  


Define album filter function and test function

## Converting song duration and finding albums over a length threshold

In [8]:
ts_songs['duration_sec'] = ts_songs['duration_ms'] / 1000
ts_songs['duration_min'] = ts_songs['duration_sec'] / 60

print(ts_songs[['track_name', 'duration_ms', 'duration_min']].head())

                             track_name  duration_ms  duration_min
0         Fortnight (feat. Post Malone)       228965      3.816083
1         The Tortured Poets Department       293048      4.884133
2  My Boy Only Breaks His Favorite Toys       203801      3.396683
3                              Down Bad       261228      4.353800
4                       So Long, London       262974      4.382900


Created new duration columns

In [9]:
def find_albums_by_time(input_df, input_min_duration):
    total_by_album = input_df.groupby("album")["duration_min"].sum().reset_index()
    long_albums = total_by_album[total_by_album["duration_min"] > input_min_duration]
    return long_albums

albums_80 = find_albums_by_time(ts_songs, 80)
print("Albums longer than 80 minutes:")
print(albums_80)

Albums longer than 80 minutes:
                                                album  duration_min
3                    1989 (Taylor's Version) [Deluxe]     81.301667
5                         Fearless (Taylor's Version)    106.541500
11                   Midnights (The Til Dawn Edition)     80.586083
13                               Red (Deluxe Edition)     90.217667
14                             Red (Taylor's Version)    130.663833
16                         Speak Now (Deluxe Edition)     91.840117
17                       Speak Now (Taylor's Version)    104.734183
21       The Tortured Poets Department: The Anthology    122.645733
26  folklore: the long pond studio sessions (from ...    134.704517
28     reputation Stadium Tour Surprise Song Playlist    186.266467


Defining the function to find albums by total time. Group by album and sum duration and filter for albums longer than the threshold

In [10]:
albums_120 = find_albums_by_time(ts_songs, 120)
print("Albums longer than 120 minutes:")
print(albums_120)

Albums longer than 120 minutes:
                                                album  duration_min
14                             Red (Taylor's Version)    130.663833
21       The Tortured Poets Department: The Anthology    122.645733
26  folklore: the long pond studio sessions (from ...    134.704517
28     reputation Stadium Tour Surprise Song Playlist    186.266467


Now tested with 120 minutes instead of 80 minutes

## Classifying albums as short, medium or long by total duration

In [11]:
lower_threshold = 30
upper_threshold = 80

We set upper and lower thresholds

In [12]:
total_by_album = ts_songs.groupby("album")["duration_min"].sum().reset_index()

def my_classify(my_time_length):
    if my_time_length < lower_threshold:
        return "EP/Short Album"
    elif my_time_length > upper_threshold:
        return "Long Album/Anthology"
    else:
        return "Standard Album"

total_by_album["category"] = total_by_album["duration_min"].apply(my_classify)
print(total_by_album.head())

                              album  duration_min              category
0                              1989     48.797733        Standard Album
1             1989 (Deluxe Edition)     68.760800        Standard Album
2           1989 (Taylor's Version)     77.972117        Standard Album
3  1989 (Taylor's Version) [Deluxe]     81.301667  Long Album/Anthology
4                          Fearless     53.547900        Standard Album


Building logic outside function structure first

In [13]:
# Defining the comprehensive function
def albums_by_length(df, lower_threshold, upper_threshold):
    total_by_album = df.groupby("album")["duration_min"].sum().reset_index()

    def classify(duration):
        if duration < lower_threshold:
            return "short_album"
        elif duration > upper_threshold:
            return "long_album"
        else:
            return "standard_album"

    total_by_album["category"] = total_by_album["duration_min"].apply(classify)
    return total_by_album

# Testing the function
test_results = albums_by_length(ts_songs, lower_threshold=30, upper_threshold=80)
print(test_results.head())

                              album  duration_min        category
0                              1989     48.797733  standard_album
1             1989 (Deluxe Edition)     68.760800  standard_album
2           1989 (Taylor's Version)     77.972117  standard_album
3  1989 (Taylor's Version) [Deluxe]     81.301667      long_album
4                          Fearless     53.547900  standard_album


Defining the comprehensive function and testing

## Cleaning video counts and aggregating by YouTube category

In [14]:
youtubers = pd.read_csv('/anvil/projects/tdm/data/youtube/most_subscribed_youtube_channels.csv')

youtubers['video_count2'] = pd.to_numeric(youtubers['video count'].str.replace(",", "", regex=False))

Reading the data set and cleaning the video count

In [16]:
video_count_per_genre = youtubers.groupby("category")["video_count2"].sum()
print(video_count_per_genre)

category
Autos & Vehicles            2874
Comedy                     93562
Education                 124727
Entertainment            2674176
Film & Animation          133319
Gaming                    427292
Howto & Style              81419
Movies                      5576
Music                     510337
News & Politics          2754693
Nonprofits & Activism     188445
People & Blogs           1045091
Pets & Animals             23960
Science & Technology       36622
Shows                     283027
Sports                    140464
Trailers                   13613
Travel & Events              632
Name: video_count2, dtype: int64


Grouping youtubers by category

In [17]:
gm_subset = youtubers[youtubers["category"].isin(["Gaming", "Music"])]

grouped_counts = gm_subset.groupby(["category", "started"])["video_count2"].sum().unstack()
print(grouped_counts)

started      2005      2006     2007     2008     2009     2010     2011  \
category                                                                   
Gaming    17878.0  163116.0  11967.0  16356.0  15342.0   9998.0  27421.0   
Music      1446.0   36423.0  51621.0  33451.0  16392.0  31766.0  89353.0   

started      2012     2013      2014    2015    2016    2017    2018    2019  \
category                                                                       
Gaming    72858.0  32267.0   31332.0  7342.0  7903.0  9302.0  2952.0  1126.0   
Music     44299.0  71681.0  119692.0  5447.0  8477.0   156.0   121.0    12.0   

started    2020  
category         
Gaming    132.0  
Music       NaN  


Filter for specific categories and group by two columns

## Finding the top YouTuber by subscribers within a genre

In [18]:
youtubers['subscribers2'] = pd.to_numeric(youtubers['subscribers'].str.replace(",", "", regex=False))

def top_youtuber_by_genre(input_df, genre):
    genre_rows = input_df[input_df["category"] == genre]
    top_index = genre_rows['subscribers2'].idxmax()
    return genre_rows.loc[top_index]

Clean the subscribers column and make a function to find top youtuber in a genre

In [19]:
top_gaming = top_youtuber_by_genre(youtubers, "Gaming")
print("Top Gaming YouTuber:")
print(top_gaming[['Youtuber', 'subscribers2']])

Top Gaming YouTuber:
Youtuber        PewDiePie
subscribers2    111000000
Name: 5, dtype: object


Testing with Gaming and then Music

In [20]:
top_music = top_youtuber_by_genre(youtubers, "Music")
print("\nTop Music YouTuber:")
print(top_music[['Youtuber', 'subscribers2']])


Top Music YouTuber:
Youtuber         T-Series
subscribers2    222000000
Name: 0, dtype: object


This dataset from 3 years ago lists PewDiePie as the top gaming creator. However, the landscape has shifted significantly by 2026. As mentioned in the context, MrBeast has since surpassed all individual creators, reaching over 400 million subscribers. While the categories (Music, Gaming, etc.) remain relevant, the scale of subscriber counts has grown exponentially.

## Pledge

By submitting this work I hereby pledge that this is my own, personal work. I've acknowledged in the designated place at the top of this file all sources that I used to complete said work, including but not limited to: online resources, books, and electronic communications. I've noted all collaboration with fellow students and/or TA's. I did not copy or plagiarize another's work.

> As a Boilermaker pursuing academic excellence, I pledge to be honest and true in all that I do. Accountable together – We are Purdue.

https://www.purdue.edu/odos/osrr/honor-pledge/
